# GDB-11 `.tgz` → ECFP4 → Tanimoto Candidates

This notebook reads the uploaded `gdb11.tgz` archive directly. It does **not** require manually extracting `gdb11_size01.smi`.

The archive contains `gdb11_size01.smi` through `gdb11_size11.smi`. The notebook streams these files in chunks, computes 1024-bit ECFP4 fingerprints (`radius=2`) with DeepChem, finds all GDB-11 molecules with Tanimoto similarity > 0.5 to the generated fingerprints, outputs the actual SMILES, and converts the selected SMILES back to ECFP4.

In [2]:
import os
import tarfile
import numpy as np
import pandas as pd
import deepchem as dc

GDB11_ARCHIVE = "gdb11.tgz"
GENERATED_FILE = "gen_ESOL_class1_cont.csv"
TANIMOTO_THRESHOLD = 0.5
CHUNK_SIZE = 100_000

print("GDB-11 archive:", GDB11_ARCHIVE)
print("Archive exists:", os.path.exists(GDB11_ARCHIVE))

if not os.path.exists(GDB11_ARCHIVE):
    raise FileNotFoundError(
        f"GDB-11 archive not found: {GDB11_ARCHIVE}. "
        "Place gdb11.tgz in the notebook working directory."
    )


GDB-11 archive: gdb11.tgz
Archive exists: True


## 1. Inspect the archive

This confirms which SMILES files are actually inside `gdb11.tgz`.

In [3]:
with tarfile.open(GDB11_ARCHIVE, "r:gz") as tar:
    smi_members = [
        m for m in tar.getmembers()
        if m.isfile() and m.name.lower().endswith(".smi")
    ]

print("SMILES files in archive:")
for m in smi_members:
    print(" ", m.name)

if not smi_members:
    raise FileNotFoundError("No .smi files were found inside gdb11.tgz")


SMILES files in archive:
  gdb11_size01.smi
  gdb11_size02.smi
  gdb11_size03.smi
  gdb11_size04.smi
  gdb11_size05.smi
  gdb11_size06.smi
  gdb11_size07.smi
  gdb11_size08.smi
  gdb11_size09.smi
  gdb11_size10.smi
  gdb11_size11.smi


## 2. ECFP4: 1024 bits, radius = 2

In [4]:
ecfp_featurizer = dc.feat.CircularFingerprint(
    size=1024,
    radius=2
)

print("ECFP size:", 1024)
print("ECFP radius:", 2)


ECFP size: 1024
ECFP radius: 2


## 3. Load generated ECFP fingerprints

In [5]:
generated_df = pd.read_csv(GENERATED_FILE)

ecfp_columns = [
    c for c in generated_df.columns
    if c.startswith("ECFP_")
]

X_generated = generated_df[ecfp_columns].to_numpy(dtype=np.int8)

print("Generated fingerprint shape:", X_generated.shape)
assert X_generated.shape[1] == 1024, "Expected 1024 ECFP columns"


Generated fingerprint shape: (1000, 1024)


## 4. Tanimoto similarity

In [6]:
def tanimoto_similarity(query_fp, reference_fps):
    query_fp = np.asarray(query_fp, dtype=np.int8)
    reference_fps = np.asarray(reference_fps, dtype=np.int8)

    intersection = np.sum(reference_fps * query_fp, axis=1)
    query_count = np.sum(query_fp)
    reference_count = np.sum(reference_fps, axis=1)
    union = query_count + reference_count - intersection

    return np.divide(
        intersection,
        union,
        out=np.zeros_like(intersection, dtype=float),
        where=union != 0
    )


## 5. Stream every GDB-11 `.smi` file from the archive

The molecules are processed in chunks, so the notebook does not create one giant 26-million-molecule SMILES list or fingerprint matrix.

In [7]:
def iter_smiles_chunks(file_obj, chunk_size=100_000):
    """Yield SMILES chunks from an open GDB .smi member."""
    chunk = []

    for raw_line in file_obj:
        if isinstance(raw_line, bytes):
            line = raw_line.decode("utf-8", errors="ignore").strip()
        else:
            line = raw_line.strip()

        if not line:
            continue

        # GDB .smi format: SMILES followed by additional tab-separated fields.
        smiles = line.split()[0]
        chunk.append(smiles)

        if len(chunk) >= chunk_size:
            yield chunk
            chunk = []

    if chunk:
        yield chunk


## 6. Find the top 5 GDB-11 molecules for every generated sample

Instead of applying a Tanimoto threshold, this keeps the **five highest-similarity molecules for each generated fingerprint** across the entire GDB-11 archive. The top-five list is updated chunk by chunk, so only a small number of candidates per generated sample are retained in memory.

In [8]:
# Keep only the best 5 candidates per generated molecule across all chunks.
TOP_K = 5
top_candidates = [[] for _ in range(len(X_generated))]
global_index = 0

with tarfile.open(GDB11_ARCHIVE, "r:gz") as tar:
    for member_number, member in enumerate(smi_members, start=1):
        print(f"\nProcessing {member.name} ({member_number}/{len(smi_members)})")

        extracted = tar.extractfile(member)
        if extracted is None:
            print("Could not read:", member.name)
            continue

        file_count = 0

        for smiles_chunk in iter_smiles_chunks(extracted, CHUNK_SIZE):
            fingerprints_chunk = np.asarray(
                ecfp_featurizer.featurize(smiles_chunk), dtype=np.int8
            )
            chunk_indices = np.arange(global_index, global_index + len(smiles_chunk))

            for gen_idx, generated_fp in enumerate(X_generated):
                similarities = tanimoto_similarity(generated_fp, fingerprints_chunk)

                # Only the five best candidates from a chunk can enter the global top five.
                local_k = min(TOP_K, len(similarities))
                if local_k == 0:
                    continue
                local_top_idx = np.argpartition(similarities, -local_k)[-local_k:]

                merged = top_candidates[gen_idx]
                for local_idx in local_top_idx:
                    local_idx = int(local_idx)
                    merged.append({
                        "generated_index": gen_idx,
                        "gdb11_index": int(chunk_indices[local_idx]),
                        "tanimoto_similarity": float(similarities[local_idx]),
                        "smiles": smiles_chunk[local_idx],
                    })

                merged.sort(key=lambda row: row["tanimoto_similarity"], reverse=True)
                top_candidates[gen_idx] = merged[:TOP_K]

            global_index += len(smiles_chunk)
            file_count += len(smiles_chunk)

        print(f"Molecules processed in {member.name}: {file_count:,}")

print(f"\nTotal GDB-11 molecules processed: {global_index:,}")
print(f"Generated samples: {len(X_generated):,}")
print(f"Maximum matches retained: {len(X_generated) * TOP_K:,}")


Processing gdb11_size01.smi (1/11)
Molecules processed in gdb11_size01.smi: 4

Processing gdb11_size02.smi (2/11)
Molecules processed in gdb11_size02.smi: 9

Processing gdb11_size03.smi (3/11)
Molecules processed in gdb11_size03.smi: 20

Processing gdb11_size04.smi (4/11)
Molecules processed in gdb11_size04.smi: 80

Processing gdb11_size05.smi (5/11)
Molecules processed in gdb11_size05.smi: 352

Processing gdb11_size06.smi (6/11)
Molecules processed in gdb11_size06.smi: 1,850

Processing gdb11_size07.smi (7/11)
Molecules processed in gdb11_size07.smi: 10,568

Processing gdb11_size08.smi (8/11)
Molecules processed in gdb11_size08.smi: 66,706

Processing gdb11_size09.smi (9/11)
Molecules processed in gdb11_size09.smi: 444,313

Processing gdb11_size10.smi (10/11)
Molecules processed in gdb11_size10.smi: 3,114,041

Processing gdb11_size11.smi (11/11)
Molecules processed in gdb11_size11.smi: 22,796,628

Total GDB-11 molecules processed: 26,434,571
Generated samples: 1,000
Maximum matches r

## 7. Top-five candidate table

In [9]:
results = [
    candidate
    for candidates_for_sample in top_candidates
    for candidate in candidates_for_sample
]

matches_df = pd.DataFrame(
    results,
    columns=[
        "generated_index",
        "gdb11_index",
        "tanimoto_similarity",
        "smiles",
    ]
)

if not matches_df.empty:
    matches_df = matches_df.sort_values(
        ["generated_index", "tanimoto_similarity"],
        ascending=[True, False]
    ).reset_index(drop=True)
    matches_df["rank"] = matches_df.groupby("generated_index").cumcount() + 1

print("Generated molecules:", len(X_generated))
print("GDB-11 molecules searched:", global_index)
print("Top K per generated sample:", TOP_K)
print("Total candidate matches:", len(matches_df))

matches_df.head(20)

Generated molecules: 1000
GDB-11 molecules searched: 26434571
Top K per generated sample: 5
Total candidate matches: 5000


,generated_index,gdb11_index,tanimoto_similarity,smiles,rank
0,0,15,0.444444,CCO,1
1,0,13,0.375000,CCC,2
2,0,52,0.363636,CCCO,3
3,0,1975686,0.363636,CCC(CC)=C(CC)CC,4
4,0,50,0.333333,CCCC,5
5,1,194,0.583333,CNC(C)=O,1
6,1,2442,0.583333,CC(=O)C(C)(C)C,2
7,1,13256,0.533333,CNC(=O)C(C)(C)C,3
8,1,14592,0.533333,CNC(C)(C)C(C)=O,4
9,1,41,0.500000,CC(C)=O,5


## 8. Save top-five SMILES and similarities

In [15]:
MATCHES_FILE = "GDB11_top5_Tanimoto_per_generated_sample.csv"

matches_df.to_csv(MATCHES_FILE, index=False)
#print("Saved:", MATCHES_FILE)

## 9. Transform the top-five candidate SMILES back into ECFP4

In [16]:
if matches_df.empty:
    print("No candidate rows were produced.")
else:
    candidate_smiles = matches_df["smiles"].tolist()
    candidate_ecfp = ecfp_featurizer.featurize(candidate_smiles)
    candidate_ecfp = np.asarray(candidate_ecfp, dtype=np.int8)

    print("Candidate SMILES:", len(candidate_smiles))
    print("Candidate ECFP shape:", candidate_ecfp.shape)

Candidate SMILES: 5000
Candidate ECFP shape: (5000, 1024)


## 10. Save SMILES + rank + similarity + ECFP4

In [17]:
if not matches_df.empty:
    ecfp_columns = [f"ECFP_{i}" for i in range(1024)]
    candidate_ecfp_df = pd.DataFrame(candidate_ecfp, columns=ecfp_columns)

    candidate_ecfp_df.insert(0, "tanimoto_similarity", matches_df["tanimoto_similarity"].values)
    candidate_ecfp_df.insert(0, "rank", matches_df["rank"].values)
    candidate_ecfp_df.insert(0, "generated_index", matches_df["generated_index"].values)
    candidate_ecfp_df.insert(0, "smiles", matches_df["smiles"].values)

    OUTPUT_FILE = "GDB11_top5_Tanimoto_per_generated_sample_ECFP4.csv"
    candidate_ecfp_df.to_csv(OUTPUT_FILE, index=False)

    print("Saved:", OUTPUT_FILE)
    print("Final shape:", candidate_ecfp_df.shape)

Saved: GDB11_top5_Tanimoto_per_generated_sample_ECFP4.csv
Final shape: (5000, 1028)
